In [0]:
from pyspark.sql.functions import *

orders_df = spark.read.option(
    "multiLine",
    "true"
).json(
    "/Volumes/workspace/default/lakehouse_raw_data/orders.json"
)

display(orders_df.limit(10))

print("Columns:", orders_df.columns)

amount,customer_id,order_id,order_time,status
4235.29,6425.0,1,2025-02-26 17:41:00,SUCCESS
-3554.03,5570.0,2,2025-11-29 17:15:00,FAILED
977.88,2643.0,3,2025-11-16 02:05:00,SUCCESS
4298.18,1959.0,4,2025-10-16 17:30:00,SUCCESS
603.99,4695.0,5,2025-09-02 22:49:00,SUCCESS
4365.65,7010.0,6,2025-02-19 14:54:00,SUCCESS
241.2,6543.0,7,2025-06-28 08:29:00,SUCCESS
515.81,9258.0,8,2025-01-28 00:42:00,SUCCESS
2172.64,4874.0,9,2025-03-05 12:42:00,SUCCESS
1975.64,7382.0,10,2025-11-15 21:39:00,SUCCESS


Columns: ['amount', 'customer_id', 'order_id', 'order_time', 'status']


In [0]:
# Null checks
null_customer_ids = orders_df.filter(
    col("customer_id").isNull()
).count()

null_amounts = orders_df.filter(
    col("amount").isNull()
).count()

# Duplicate checks
duplicate_orders = orders_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .count()

# Negative revenue checks
negative_amounts = orders_df.filter(
    col("amount") < 0
).count()

print("Null Customer IDs:", null_customer_ids)
print("Null Amounts:", null_amounts)
print("Duplicate Orders:", duplicate_orders)
print("Negative Amounts:", negative_amounts)

if (
    null_customer_ids == 0 and
    null_amounts == 0 and
    duplicate_orders == 0 and
    negative_amounts == 0
):
    print("DATA QUALITY PASSED")
else:
    print("DATA QUALITY FAILED")

Null Customer IDs: 300
Null Amounts: 0
Duplicate Orders: 500
Negative Amounts: 10100
DATA QUALITY FAILED


In [0]:
from pyspark.sql.functions import *

clean_orders_df = orders_df \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("amount") >= 0) \
    .dropDuplicates(["order_id"])

print("Original Count:", orders_df.count())
print("Cleaned Count:", clean_orders_df.count())

display(clean_orders_df.limit(20))

Original Count: 100500
Cleaned Count: 89693


amount,customer_id,order_id,order_time,status
2352.58,6738.0,28,2025-05-27 13:07:00,SUCCESS
4348.67,4877.0,29,2025-09-22 18:38:00,PENDING
88.78,6928.0,30,2025-09-29 12:17:00,SUCCESS
133.2,2624.0,33,2025-01-11 15:00:00,SUCCESS
2707.09,4774.0,93,2025-07-15 17:40:00,SUCCESS
1395.15,4081.0,151,2025-12-18 20:56:00,SUCCESS
2228.25,3935.0,171,2025-05-17 02:35:00,SUCCESS
2762.45,408.0,200,2025-01-11 20:00:00,SUCCESS
4673.77,1216.0,221,2025-01-08 13:43:00,SUCCESS
4317.7,7312.0,224,2025-05-23 03:36:00,SUCCESS


In [0]:
null_customer_ids = clean_orders_df.filter(
    col("customer_id").isNull()
).count()

null_amounts = clean_orders_df.filter(
    col("amount").isNull()
).count()

duplicate_orders = clean_orders_df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .count()

negative_amounts = clean_orders_df.filter(
    col("amount") < 0
).count()

print("Null Customer IDs:", null_customer_ids)
print("Null Amounts:", null_amounts)
print("Duplicate Orders:", duplicate_orders)
print("Negative Amounts:", negative_amounts)

if (
    null_customer_ids == 0 and
    null_amounts == 0 and
    duplicate_orders == 0 and
    negative_amounts == 0
):
    print("DATA QUALITY PASSED")
else:
    print("DATA QUALITY FAILED")

Null Customer IDs: 0
Null Amounts: 0
Duplicate Orders: 0
Negative Amounts: 0
DATA QUALITY PASSED
